# 03 — Tool Middleware

Exa returns raw HTML that can blow context windows. AG2's `ToolMiddleware` intercepts tool results **after execution, before they hit the stream** — the right place for cleaning and truncation.

This is how lionag2 keeps search results usable without blanket message truncation.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## The problem

Even with `max_characters=3000` on the API side, Exa can return content with embedded scripts, styles, and HTML tags. Without cleaning, a single search result can consume 20K+ tokens of context.

In [ ]:
from lionag2.research.middleware import MAX_RESULT_CHARS, _clean_html

raw_html = """
<html><head><script>var x = 1;</script>
<style>.cls { color: red; }</style></head>
<body><h1>Paper Title</h1>
<p>We show that spin fluctuations &amp; pairing mechanisms...</p>
<div class="sidebar">Navigation links</div>
</body></html>
"""

cleaned = _clean_html(raw_html)
print(f"Raw:     {len(raw_html)} chars")
print(f"Cleaned: {len(cleaned)} chars")
print(f"Result:  {cleaned!r}")
print(f"\nMax result chars before truncation: {MAX_RESULT_CHARS:,}")

## The middleware pattern

AG2's `ToolMiddleware` wraps the tool execution chain. lionag2's `clean_search_results` is passed to `ExaToolkit(middleware=(...))` — it runs after every search/fetch call:

```python
exa = ExaToolkit(
    num_results=5,
    max_characters=5000,
    middleware=(clean_search_results,),  # <-- intercepts results
)
```

The middleware receives the `ToolCallEvent` and a `call_next` function. It calls `call_next` to get the raw result, then cleans each text part.

In [ ]:
import inspect

from lionag2.research.middleware import clean_search_results

print(inspect.getsource(clean_search_results))

## How the engine uses it

In `ResearchEngine._resolve_tools()`, each agent gets a **fresh** ExaToolkit instance with the middleware installed. Fresh instances avoid deepcopy issues with shared HTTP clients:

```python
exa = ExaToolkit(
    num_results=5,
    max_characters=5000,
    middleware=(clean_search_results,),
)
tools.extend(exa.tools)
```

This is cheaper than truncating at the prompt level — the cleaning happens once at the tool boundary, and every downstream consumer (the agent, the stream, the knowledge store) sees clean text.

## Up next

Events flow through each agent's stream. Tutorial 04 introduces the Flow architecture — a shared Pile with named Progression streams, and bridge observers that forward typed events for reactive coordination.